# 01 · Target Factory — the targets, the panel and the model input

**What this stage is for.** It builds the targets of one use case, or of several that share the
entity's feature sources, keys and period (`time.period`: month, week or day), each from its
target source by the rule in its `data.target` (or from an upstream column). It joins the sources
into one panel and saves them as one **model input**: the sources joined plus one column
`target_<use_case>` per use case (`docs/components/dataset.md`, D-131). Each later stage reads
that table with its own config, keeps its own target as `target`, drops the others and derives
the rest on read: eligibility, the candidates, the contract checks and the client split. Those
may change without a rebuild; saving freezes each config's data part (sources, keys, period,
target), and later changes go through the decision audit log (`record_decision`, D-125).

**The example runs two use cases** on the synthetic data: `cc_acquisition` (credit card) and
`td_acquisition` (term deposit), one model input with a target column each. A use-case repo lists
its own configs in the setup cell; one config is fine. The CLI does the same:
`python -m core_modelling run a.yml b.yml --stages target_factory`.

**What to look at.**
1. **The target summaries** (step 1), one per use case: rows per period; `null_share`, the
   targets unknown at the panel edge, for leavers and across gaps; `null_value_share`, the share
   of complete windows that held a null in the target column (a null never counts as an event);
   and, for the observation window only, the mean target, the positive rows and the positive
   events. A rate that drifts across the window, or a null share above a few percent, is worth a
   question upstream before anything is modelled.
2. **The join printout** (step 2): the share of the spine's rows each source matched per period
   and the null share per target. A source that matches few rows usually means a key or a
   period tag that differs between sources. The saved table lists one target column per use case.
3. **The window frames** (step 3): what EDA will see for each use case, read back from the same
   table with that use case's eligibility, contract and split lines printing as they are
   derived. A filter that drops nothing, or almost everything, usually means a wrong condition.

**Decisions to record:** none here. The data definition is the config itself: if a target is
wrong, change its config and rebuild; if the eligibility or the candidates are wrong, change the
config and rerun from EDA, the table stays. The data-definition hash changes either way.

**Docs:** `docs/components/target_factory.md` · `docs/components/dataset.md` ·
`docs/components/config.md` · the next stage: `02_eda` (M2.2).


In [ ]:
from pathlib import Path

import pandas as pd

import core_modelling
from core_modelling.config import load_config
from core_modelling.dataset import build_frames, save_model_input
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.target_factory import run_target_factory

# The use cases and where the framework runs: absolute paths, never relative to the working
# directory (T5). The repo's examples are found from the editable install; a use-case repo writes
# its own absolute config paths here and reads the environment from $CORE_MODELLING_ENVIRONMENT.
# The configs share the entity's feature sources, keys, period and data.model_input.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIGS = [REPO / "conf/base/cc_acquisition.yml", REPO / "conf/base/td_acquisition.yml"]
ENTITY = "entity_a"
ENVIRONMENT = REPO / "conf/local/environment.yml"  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(str(ENVIRONMENT) if ENVIRONMENT else None)
spark = get_spark_session(env)
cfgs = [load_config(path) for path in CONFIGS]
for cfg in cfgs:
    print(cfg.use_case, cfg.task, cfg.data.target)
    print(cfg.time)

## 1. The targets
One summary per use case, over every row of its target source, before eligibility. For each,
check the per-period rows and null shares first, then the window periods' rate: the `all` row
pools the window. Two products on the same clients usually have different rates; each should be
plausible for its product and steady across the window.


In [ ]:
summaries = {cfg.use_case: run_target_factory(cfg, ENTITY, env, spark) for cfg in cfgs}

## 2. Save the model input
The sources are joined once (the join lines print as it goes) and written as **one table with a
target column per use case** to the path the environment file maps for `data.model_input`, with
`_model_input.json` in the same folder (the shared sources, keys and period, the periods covered,
the join log and, per use case, the target-spec hash, the lineage, the window and the null
share). Nothing use-case-specific beyond the targets is saved. From here on each config's data
part is frozen; a use case added later means saving again with the longer list.


In [ ]:
path = save_model_input(cfgs, ENTITY, env, spark)

In [ ]:
[column for column in spark.read.parquet(path).columns if column.startswith("target_")]

## 3. What EDA will see, per use case
Each use case reads the same table with its own config: it keeps its own target as `target`,
drops the other target columns, and derives its eligibility, contract and split on read (the lines
print per use case). Every client after its analysis filters, with `unit_id`, `period`, the
candidates, its modelling tags (here `non_holders`, a different filter per product), `eligible`,
`target` and `split` as columns. The table below compares them: rows, candidate columns, the
share of eligible rows and the target rate on them. The rows differ because `cc_acquisition`
drops the clients with neither a known income nor a salary transfer and `td_acquisition` keeps
everyone. Every use case's frame is held in memory at once, so with several large use cases
build one at a time.


In [ ]:
frames = {cfg.use_case: build_frames(cfg, ENTITY, env, spark, stage="eda") for cfg in cfgs}
pd.DataFrame.from_dict(
    {
        use_case: {
            "rows": len(f.window),
            "candidate columns": len(f.meta["candidates"]),
            "eligible share": f.window["eligible"].mean(),
            "target mean (eligible)": f.window.loc[f.window["eligible"] == 1, "target"].mean(),
        }
        for use_case, f in frames.items()
    },
    orient="index",
)

One use case's frame in detail: the share tagged per period is its eligibility funnel. Change
`USE_CASE` to look at the other.


In [ ]:
USE_CASE = "cc_acquisition"
window = frames[USE_CASE].window
print(USE_CASE, "lineage excluded:", frames[USE_CASE].meta["lineage"])
window.groupby("period")[[*frames[USE_CASE].meta["tags"], "eligible"]].mean()

In [ ]:
window.head()